In [55]:
!pip install pandas -q
!pip install duckdb -q

In [56]:
import pandas as pd
import duckdb

In [57]:
data_dir = "../csv"  
ai="grok"

sales = pd.read_csv(f"{data_dir}/sales-{ai}.csv", parse_dates=['DATE'])
parts = pd.read_csv(f"{data_dir}/parts.csv")
parts['SQFT'] = parts['SQFT'].astype('Int64')

f"rows - parts: {len(parts)}, sales: {len(sales)}"

'rows - parts: 7, sales: 877'

In [58]:
duckdb.query("""
    DROP VIEW IF EXISTS sales_monthly_view;
    
    CREATE VIEW sales_monthly_view AS
    SELECT 
        strftime('%Y-%m', date) AS month,
        supplier,
        licensee,
        part_id,
        SUM(qty) AS tot_qty,
        SUM(amt) AS tot_amt
    FROM sales
    GROUP BY month, supplier, licensee, part_id
""")

monthly_sales_view = duckdb.query("SELECT * FROM sales_monthly_view ORDER BY month, supplier, licensee, part_id")
monthly_sales = monthly_sales_view.df()

monthly_sales

,month,SUPPLIER,LICENSEE,PART_ID,tot_qty,tot_amt
0,2023-03,nippon-metal,ninja-roofing,glass-clip,109.0,128600.0
1,2023-03,nippon-metal,ninja-roofing,glass-decoration,114.0,86100.0
2,2023-03,nippon-metal,ninja-roofing,gold-clip,156.0,89500.0
3,2023-03,nippon-metal,ninja-roofing,gold-decoration,165.0,64400.0
4,2023-03,nippon-metal,ninja-roofing,roof-polish,57.0,11100.0
...,...,...,...,...,...,...
872,2026-01,nippon-metal,rice-roofers,tin-decoration,144.0,16300.0
873,2026-01,us-steel,ranger-roofing,glass-clip,142.0,178100.0
874,2026-01,us-steel,ranger-roofing,gold-clip,120.0,75800.0
875,2026-01,us-steel,ranger-roofing,roof-polish,117.0,27800.0


In [59]:
# make view
duckdb.query("""
    DROP VIEW IF EXISTS monthly_supplier_totals;

    CREATE VIEW monthly_supplier_totals AS
    SELECT 
        strftime('%Y-%m', date) AS month,
        supplier,
        SUM(amt) AS tot_amt
    FROM sales
    GROUP BY month, supplier
    ORDER BY month, supplier
""")


print("\n\nSupplier Monthly Totals\n")

# see view
duckdb.query("""
    SELECT 
        month,
        supplier,
        printf('%,d', tot_amt) as tot_amt
    FROM monthly_supplier_totals
    ORDER BY month, supplier
""").df()



Supplier Monthly Totals



,month,SUPPLIER,tot_amt
0,2023-03,nippon-metal,"709,400"
1,2023-03,us-steel,"569,200"
2,2023-04,nippon-metal,"260,900"
3,2023-04,us-steel,"1,229,500"
4,2023-05,nippon-metal,"629,100"
...,...,...,...
60,2025-11,us-steel,"1,164,100"
61,2025-12,nippon-metal,"245,200"
62,2025-12,us-steel,"500,500"
63,2026-01,nippon-metal,"1,153,000"


In [60]:
duckdb.query("""
    DROP VIEW IF EXISTS monthly_licensee_totals;

    CREATE VIEW monthly_licensee_totals AS
    SELECT 
        strftime('%Y-%m', date) AS month,
        licensee,
        SUM(amt) AS tot_amt
    FROM sales
    GROUP BY month, licensee
    ORDER BY month, licensee
""")
from IPython.display import display

print("\n\nLicensee Monthly Totals\n")

duckdb.query("""
    SELECT 
        month,
        licensee,
        printf('%,d', tot_amt) as tot_amt
    FROM monthly_licensee_totals
    ORDER BY month, licensee
""").df()



Licensee Monthly Totals



,month,LICENSEE,tot_amt
0,2023-03,global-roof,"417,500"
1,2023-03,ninja-roofing,"399,500"
2,2023-03,ranger-roofing,"151,700"
3,2023-03,rice-roofers,"309,900"
4,2023-04,global-roof,"387,900"
...,...,...,...
140,2025-12,zztop-roof,"338,300"
141,2026-01,global-roof,"338,100"
142,2026-01,ninja-roofing,"261,200"
143,2026-01,ranger-roofing,"306,300"
